# HHEM-2.1-open — Colab T4 compatibility smoke test (M7)

**Status: PENDING.** This notebook has not been run yet. Run it once on a Colab **T4** and attach the downloaded report to the PR.

**What it does**
- Uses the merged scorer `ml/hhem-baseline/hhem_score.py` and its pins: HHEM at `de9a5e6b`, flan-t5-base at `7bcac572`, all files checked by SHA-256 and loaded offline from clean folders.
- Checks that HHEM loads with **transformers 4.56.2**.
- Scores the **7 public examples** from the pinned model card and compares them with the published scores (tolerance 0.005, same > 0.5 decisions).
- Records the GPU, driver, CUDA, library versions, model load time and per-example latency.

**What it does not do:** it reads no M7 data, does no dev scoring, no fine-tuning and no threshold selection.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*, then *Connect*. Any other GPU or a CPU runtime makes the script stop with exit code 2 before it downloads anything.
2. Use a **fresh** runtime (*Runtime → Disconnect and delete runtime*), so no other transformers version is already loaded.

**Run:** *Runtime → Run all*. It takes about 3–5 minutes, mostly the 438 MB HHEM weights download. Then download the report from the last cell.

In [ ]:
# 1. Confirm the GPU (it must say "Tesla T4").
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## 2. Get the code
The repository is public, so no token is needed. For an exactly reproducible run, set `REF` to the commit hash given in the PR. `main` works once the PR is merged.

In [ ]:
REF = "main"  # or the exact commit hash from the PR
REPO = "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS.git"
!rm -rf /content/Enterprise-AI-OS
!git clone --quiet {REPO} /content/Enterprise-AI-OS
!cd /content/Enterprise-AI-OS && git checkout --quiet {REF} && git log -1 --format="commit %H  %s"
%cd /content/Enterprise-AI-OS/ml/hhem-baseline
!ls

## 3. Install the pinned libraries
The pinned libraries are `transformers==4.56.2`, `tokenizers==0.22.2` and `safetensors==0.8.0`, among others. torch is Colab's own CUDA build. pip may warn about other Colab packages; that is expected and does not affect this test.

In [ ]:
!pip install -q -r requirements.txt
!pip show transformers tokenizers safetensors torch | grep -E "^(Name|Version)"

## 4. Run the smoke test
It runs in a fresh Python process, so the scorer can set its private module cache before transformers is imported. Exit codes: **0** passed, **1** scores or decisions differ, **2** unsupported runtime or versions.

In [ ]:
!python smoke_model_card.py --out /content/hhem-smoke-report.json; echo "exit code: $?"

## 5. Inspect the report

In [ ]:
import json

with open("/content/hhem-smoke-report.json") as fh:
    r = json.load(fh)
print("status:", r["status"], "|", r.get("reason", ""))
if "comparison" in r:
    b, c = r["backend"], r["comparison"]
    print("GPU:", r["runtime"]["device_name"], "| driver:", r.get("nvidia_driver"))
    print("torch:", b["torch"], "| CUDA:", b["cuda"], "| versions:", r["versions"])
    print("code commit:", r["code_commit"], "| model load (s):", round(r["load_seconds"], 1))
    print("latency per example:", r["latency_per_example"])
    print(f"max |actual - published| = {c['max_abs_diff']:.2e} (tolerance {c['tolerance']})")
    for e in c["examples"]:
        ok = e["within_tolerance"] and e["decision_matches"]
        print(f"{e['actual']:.6f} vs {e['published']:.6f}  ok={ok}  | {e['hypothesis']}")

## 6. Download the report
Attach `hhem-smoke-report.json` to the PR, or commit it under `docs/evidence/m7/`. It contains only the public examples, the versions and the timings.

In [ ]:
from google.colab import files

files.download("/content/hhem-smoke-report.json")